In [45]:
from pathlib import Path
import pandas as pd
import sys

ROOT = Path.cwd()

if not (ROOT / "Dirigo").exists():
    ROOT = ROOT.parent

assert (ROOT / "Dirigo").exists(), "Không tìm thấy thư mục Dirigo"
assert (ROOT / "DOCEL").exists(), "Không tìm thấy thư mục DOCEL"

print("Python:", sys.executable)
print("Project root:", ROOT.resolve())

Python: d:\Code\Dirigo\.venv\Scripts\python.exe
Project root: D:\Code\Dirigo


In [46]:
import pandas as pd

In [47]:
events_df = pd.read_csv(ROOT / "Dirigo" / "OCEL_Events.csv")
e2o_df = pd.read_csv(ROOT / "Dirigo" / "OCEL_E2O.csv")

In [48]:
prefix_map = {
    'Pcp': 'PickupPlan',
    'tr': 'Truck',
    'Cr': 'Cargo'
}
e2o_df['prefix'] = e2o_df['Object_id'].str.extract(r'^([A-Za-z]+)')
e2o_df['object_type'] = e2o_df['prefix'].map(prefix_map)


In [49]:
events_with_objects = events_df.copy()
object_types = e2o_df['object_type'].dropna().unique()
for obj_type in object_types:
    related = e2o_df[e2o_df['object_type'] == obj_type]
    grouped = related.groupby('Event_id')['Object_id'].apply(list).reset_index()
    grouped.rename(columns={'Object_id': obj_type}, inplace=True)
    events_with_objects = events_with_objects.merge(grouped, how='left', on='Event_id')

In [50]:
events_with_objects.to_csv(ROOT / "DOCEL" /"DOCEL_Events.csv", index=False)

In [51]:
truck_df = pd.read_csv(ROOT / "Dirigo" / "OCEL_Truck.csv")

In [52]:
# truck_df.rename(columns={
#     'PickupPlanID': 'Pickup Plan ID',
#     'CargoID': 'Cargo ID'
# }, inplace=True)

In [53]:
def extract_static_objects_strict(static_df):
  
    dynamic_attrs = set(static_df["Ocel_changed_field"].dropna().unique())

 
    exclude_cols = dynamic_attrs.union({"Ocel_changed_field", "Timestamp"})
    static_cols = [col for col in static_df.columns if col not in exclude_cols]


    static_objects_df = static_df[static_cols].drop_duplicates(subset=["Object_id"]).copy()


    static_objects_df = static_objects_df.rename(columns={"Object_id": "ObjectId"})

    return static_objects_df

In [54]:
docel_truck = extract_static_objects_strict(truck_df)
docel_truck.rename(columns={'ObjectId': 'Truck ID'}, inplace=True)
docel_truck

,Truck ID,LPT,Axles
0,tr46,271FSE,6.0
1,tr28,157LCX,4.0
2,tr5,012CRA,6.0
3,tr27,764KIO,4.0
4,tr41,055DSW,4.0
5,tr12,710YJQ,2.0
6,tr36,671NRX,4.0
7,tr37,280QHM,6.0
8,tr33,823MWT,6.0
9,tr49,606MXL,6.0


In [55]:
docel_truck.to_csv(ROOT / "DOCEL" /"DOCEL_Truck_Static.csv", index=False)

In [56]:
cargo_df = pd.read_csv(ROOT / "Dirigo" / "OCEL_Cargo.csv")

In [57]:
docel_cargo = extract_static_objects_strict(cargo_df)
docel_cargo.rename(columns={'ObjectId': 'Cargo ID'}, inplace=True)
docel_cargo

,Cargo ID,Cargo Type,Silo ID
0,Cr20,Corn,Sil4
1,Cr15,Rice,Sil15
2,Cr7,Rice,Sil1
3,Cr19,Other,Sil13
4,Cr3,Wheat,Sil10
5,Cr10,Corn,Sil16
6,Cr8,Rice,Sil2
7,Cr18,Other,Sil11
8,Cr14,Rice,Sil17
9,Cr11,Other,Sil19


In [58]:
docel_cargo.to_csv(ROOT / "DOCEL" /"DOCEL_Cargo_Static.csv", index=False)

In [59]:
def extract_dynamic_attribute_names(df):
    return sorted(set(df["Ocel_changed_field"].dropna().unique()))

In [60]:
truck_dynamic = extract_dynamic_attribute_names(truck_df)
truck_dynamic

['Cargo ID',
 'Pickup Plan ID',
 'Scheduled Pickup Weight',
 'Truck Status',
 'Truck Weight']

In [61]:
def generate_dynamic_attribute_table(object_df, attr_name, e2o_df, events_df):
  
    object_df = object_df.copy()
    object_df['Timestamp'] = pd.to_datetime(object_df['Timestamp'])
    events_df['Timestamp'] = pd.to_datetime(events_df['Timestamp'])

    e2o_full = e2o_df.merge(events_df, on='Event_id', how='left')
    e2o_full.rename(columns={'Timestamp': 'event_timestamp'}, inplace=True)

    target_rows = object_df[object_df['Ocel_changed_field'] == attr_name].copy()

    records = []
    for idx, row in target_rows.iterrows():
        object_id = row['Object_id']
        timestamp = row['Timestamp']
        value = row.get(attr_name)

        if pd.isna(object_id) or pd.isna(value) or pd.isna(timestamp):
            continue

        related_events = e2o_full[(e2o_full['Object_id'] == object_id) &
                                  (e2o_full['event_timestamp'] == timestamp)]

        if related_events.empty:
            continue  

        event_id = related_events.iloc[0]['Event_id']

        records.append({
            'ID': f"{attr_name}_{len(records)}",
            'value': value,
            'objectID': object_id,
            'eventID': event_id
        })

    return pd.DataFrame(records)


In [62]:
TW_df = generate_dynamic_attribute_table(truck_df, "Truck Weight", e2o_df, events_df)
TW_df.rename(columns={'ID': 'TWID','value':'Truck Weight'}, inplace=True)
TW_df.to_csv(ROOT / 'DOCEL' / 'DOCEL_TruckWeight.csv',index=False)

In [63]:
TS_df = generate_dynamic_attribute_table(truck_df, "Truck Status", e2o_df, events_df)
TS_df.rename(columns={'ID': 'TSID','value':'Truck Status'}, inplace=True)
TS_df

,TSID,Truck Status,objectID,eventID
0,Truck Status_0,Occupied,tr26,assign_tr26_Pcp44
1,Truck Status_1,Occupied,tr49,assign_tr49_Pcp44
2,Truck Status_2,Occupied,tr41,assign_tr41_Pcp87
3,Truck Status_3,Occupied,tr22,assign_tr22_Pcp87
4,Truck Status_4,Occupied,tr11,assign_tr11_Pcp48
...,...,...,...,...
195,Truck Status_195,Occupied,tr17,assign_tr17_Pcp77
196,Truck Status_196,Occupied,tr23,assign_tr23_Pcp51
197,Truck Status_197,Occupied,tr29,assign_tr29_Pcp51
198,Truck Status_198,Occupied,tr29,assign_tr29_Pcp70


In [64]:
TS_df.to_csv(ROOT / 'DOCEL' / 'DOCEL_TruckStatus.csv',index=False)

In [65]:
TCargo_df = generate_dynamic_attribute_table(truck_df, "Cargo ID", e2o_df, events_df)
TCargo_df.rename(columns={'ID': 'TCrID','value':'Cargo ID'}, inplace=True)
TCargo_df

,TCrID,Cargo ID,objectID,eventID
0,Cargo ID_0,Cr2,tr26,assign_tr26_Pcp44
1,Cargo ID_1,Cr2,tr49,assign_tr49_Pcp44
2,Cargo ID_2,Cr9,tr41,assign_tr41_Pcp87
3,Cargo ID_3,Cr9,tr22,assign_tr22_Pcp87
4,Cargo ID_4,Cr20,tr11,assign_tr11_Pcp48
...,...,...,...,...
195,Cargo ID_195,Cr13,tr17,assign_tr17_Pcp77
196,Cargo ID_196,Cr18,tr23,assign_tr23_Pcp51
197,Cargo ID_197,Cr18,tr29,assign_tr29_Pcp51
198,Cargo ID_198,Cr10,tr29,assign_tr29_Pcp70


In [66]:
TCargo_df.to_csv(ROOT / 'DOCEL' / 'DOCEL_TruckCargoID.csv',index=False)

In [ ]:
#Chỉ lấy từ bảng e2o_df (chỉ có sự kiện assign truck) còn drop truck ở bảng o2o 
TPcp_df = generate_dynamic_attribute_table(truck_df, "Pickup Plan ID", e2o_df, events_df)
TPcp_df.rename(columns={'ID': 'TPcpID','value':'Pickup Plan ID'}, inplace=True)
TPcp_df

,TPcpID,Pickup Plan ID,objectID,eventID
0,Pickup Plan ID_0,Pcp44,tr26,assign_tr26_Pcp44
1,Pickup Plan ID_1,Pcp44,tr49,assign_tr49_Pcp44
2,Pickup Plan ID_2,Pcp87,tr41,assign_tr41_Pcp87
3,Pickup Plan ID_3,Pcp87,tr22,assign_tr22_Pcp87
4,Pickup Plan ID_4,Pcp48,tr11,assign_tr11_Pcp48
...,...,...,...,...
195,Pickup Plan ID_195,Pcp77,tr17,assign_tr17_Pcp77
196,Pickup Plan ID_196,Pcp51,tr23,assign_tr23_Pcp51
197,Pickup Plan ID_197,Pcp51,tr29,assign_tr29_Pcp51
198,Pickup Plan ID_198,Pcp70,tr29,assign_tr29_Pcp70


In [68]:
TPcp_df.to_csv(ROOT / 'DOCEL' / 'DOCEL_TruckPickupplanID.csv',index=False)

In [69]:
TSchW_df = generate_dynamic_attribute_table(truck_df, "Scheduled Pickup Weight", e2o_df, events_df)
TSchW_df.rename(columns={'ID': 'TSchWID','value':'Scheduled Pickup Weight'}, inplace=True)
TSchW_df

,TSchWID,Scheduled Pickup Weight,objectID,eventID
0,Scheduled Pickup Weight_0,5369.5,tr26,assign_tr26_Pcp44
1,Scheduled Pickup Weight_1,5241.8,tr49,assign_tr49_Pcp44
2,Scheduled Pickup Weight_2,5305.6,tr41,assign_tr41_Pcp87
3,Scheduled Pickup Weight_3,5509.5,tr22,assign_tr22_Pcp87
4,Scheduled Pickup Weight_4,5476.2,tr11,assign_tr11_Pcp48
...,...,...,...,...
195,Scheduled Pickup Weight_195,5283.8,tr17,assign_tr17_Pcp77
196,Scheduled Pickup Weight_196,4941.8,tr23,assign_tr23_Pcp51
197,Scheduled Pickup Weight_197,5063.2,tr29,assign_tr29_Pcp51
198,Scheduled Pickup Weight_198,5458.8,tr29,assign_tr29_Pcp70


In [70]:
TSchW_df.to_csv(ROOT / 'DOCEL' / 'DOCEL_TruckScheduledPickupWeight.csv',index=False)

In [71]:
cargo_dynamic = extract_dynamic_attribute_names(cargo_df)
cargo_dynamic

['Cargo stock weight(scheduled)']

In [72]:
CargoSt_df = generate_dynamic_attribute_table(cargo_df, "Cargo stock weight(scheduled)", e2o_df, events_df)
CargoSt_df.rename(columns={'ID': 'CargoStID','value':'Cargo stock weight(scheduled)'}, inplace=True)
CargoSt_df

,CargoStID,Cargo stock weight(scheduled),objectID,eventID
0,Cargo stock weight(scheduled)_0,90032.0,Cr2,Lodge_Pcp44
1,Cargo stock weight(scheduled)_1,133114.7,Cr9,Lodge_Pcp87
2,Cargo stock weight(scheduled)_2,112866.2,Cr20,Lodge_Pcp48
3,Cargo stock weight(scheduled)_3,137342.6,Cr13,Lodge_Pcp55
4,Cargo stock weight(scheduled)_4,130779.5,Cr7,Lodge_Pcp69
...,...,...,...,...
95,Cargo stock weight(scheduled)_95,70689.5,Cr20,Lodge_Pcp37
96,Cargo stock weight(scheduled)_96,55629.0,Cr14,Lodge_Pcp75
97,Cargo stock weight(scheduled)_97,39322.9,Cr15,Lodge_Pcp62
98,Cargo stock weight(scheduled)_98,77221.3,Cr17,Lodge_Pcp50


In [73]:
CargoSt_df.to_csv(ROOT / 'DOCEL' / 'DOCEL_CargoStock.csv',index=False)

In [74]:
def generate_dynamic_tables_from_pickupplan(pickup_df, e2o_df, events_df):

    pickup_df = pickup_df.copy()
    pickup_df['Timestamp'] = pd.to_datetime(pickup_df['Timestamp'])
    events_df['Timestamp'] = pd.to_datetime(events_df['Timestamp'])

    e2o_full = e2o_df.merge(events_df, on='Event_id', how='left')
    e2o_full.rename(columns={'Timestamp': 'event_timestamp'}, inplace=True)

    attr_columns = [col for col in pickup_df.columns if col not in ['Object_id', 'Timestamp', 'Ocel_changed_field']]
    results = {}

    for attr in attr_columns:
        records = []

        for _, row in pickup_df.iterrows():
            object_id = row['Object_id']
            timestamp = row['Timestamp']
            value = row.get(attr)

            if pd.isna(object_id) or pd.isna(timestamp) or pd.isna(value):
                continue

            related_events = e2o_full[(e2o_full['Object_id'] == object_id) &
                                      (e2o_full['event_timestamp'] == timestamp)]
            if related_events.empty:
                continue

            event_id = related_events.iloc[0]['Event_id']


            records.append({
                'ID': 
                f"{attr}_{len(records)}",
                'value': value,
                'objectID': object_id,
                'eventID': event_id
            })

        results[attr] = pd.DataFrame(records)

    return results


In [75]:
pickup_df = pd.read_csv(ROOT / "Dirigo" / "OCEL_PickupPlan.csv")
pickup_dynamic_tables = generate_dynamic_tables_from_pickupplan(pickup_df, e2o_df, events_df)
pickup_dynamic_tables

{'CargoID':             ID value objectID      eventID
 0    CargoID_0   Cr2    Pcp44  Lodge_Pcp44
 1    CargoID_1   Cr9    Pcp87  Lodge_Pcp87
 2    CargoID_2  Cr20    Pcp48  Lodge_Pcp48
 3    CargoID_3  Cr13    Pcp55  Lodge_Pcp55
 4    CargoID_4   Cr7    Pcp69  Lodge_Pcp69
 ..         ...   ...      ...          ...
 95  CargoID_95  Cr20    Pcp37  Lodge_Pcp37
 96  CargoID_96  Cr14    Pcp75  Lodge_Pcp75
 97  CargoID_97  Cr15    Pcp62  Lodge_Pcp62
 98  CargoID_98  Cr17    Pcp50  Lodge_Pcp50
 99  CargoID_99  Cr13    Pcp77  Lodge_Pcp77
 
 [100 rows x 4 columns],
 'Num of trucks':                   ID  value objectID      eventID
 0    Num of trucks_0      2    Pcp44  Lodge_Pcp44
 1    Num of trucks_1      2    Pcp87  Lodge_Pcp87
 2    Num of trucks_2      2    Pcp48  Lodge_Pcp48
 3    Num of trucks_3      2    Pcp55  Lodge_Pcp55
 4    Num of trucks_4      2    Pcp69  Lodge_Pcp69
 ..               ...    ...      ...          ...
 95  Num of trucks_95      2    Pcp37  Lodge_Pcp37
 96  Num 

In [76]:
CargoPP_df = pickup_dynamic_tables["CargoID"]
CargoPP_df.rename(columns={'ID': 'PcpCrID','value':'Cargo ID'}, inplace=True)
CargoPP_df

,PcpCrID,Cargo ID,objectID,eventID
0,CargoID_0,Cr2,Pcp44,Lodge_Pcp44
1,CargoID_1,Cr9,Pcp87,Lodge_Pcp87
2,CargoID_2,Cr20,Pcp48,Lodge_Pcp48
3,CargoID_3,Cr13,Pcp55,Lodge_Pcp55
4,CargoID_4,Cr7,Pcp69,Lodge_Pcp69
...,...,...,...,...
95,CargoID_95,Cr20,Pcp37,Lodge_Pcp37
96,CargoID_96,Cr14,Pcp75,Lodge_Pcp75
97,CargoID_97,Cr15,Pcp62,Lodge_Pcp62
98,CargoID_98,Cr17,Pcp50,Lodge_Pcp50


In [77]:
CargoPP_df.to_csv(ROOT / 'DOCEL' / 'DOCEL_PickupplanCargo.csv',index=False)

In [78]:
Notrs_df = pickup_dynamic_tables["Num of trucks"]
Notrs_df.rename(columns={'ID': 'NotrsID','value':'Num of trucks'}, inplace=True)
Notrs_df

,NotrsID,Num of trucks,objectID,eventID
0,Num of trucks_0,2,Pcp44,Lodge_Pcp44
1,Num of trucks_1,2,Pcp87,Lodge_Pcp87
2,Num of trucks_2,2,Pcp48,Lodge_Pcp48
3,Num of trucks_3,2,Pcp55,Lodge_Pcp55
4,Num of trucks_4,2,Pcp69,Lodge_Pcp69
...,...,...,...,...
95,Num of trucks_95,2,Pcp37,Lodge_Pcp37
96,Num of trucks_96,2,Pcp75,Lodge_Pcp75
97,Num of trucks_97,2,Pcp62,Lodge_Pcp62
98,Num of trucks_98,2,Pcp50,Lodge_Pcp50


In [79]:
Notrs_df.to_csv(ROOT / 'DOCEL' / 'DOCEL_PickupplanNotrs.csv',index=False)

In [80]:
PcpTotalW_df = pickup_dynamic_tables["Total pickup weight"]
PcpTotalW_df.rename(columns={'ID': 'PcpTotalWID','value':'Total pickup weight'}, inplace=True)
PcpTotalW_df

,PcpTotalWID,Total pickup weight,objectID,eventID
0,Total pickup weight_0,10611.3,Pcp44,Lodge_Pcp44
1,Total pickup weight_1,10815.1,Pcp87,Lodge_Pcp87
2,Total pickup weight_2,11155.8,Pcp48,Lodge_Pcp48
3,Total pickup weight_3,11720.1,Pcp55,Lodge_Pcp55
4,Total pickup weight_4,10058.4,Pcp69,Lodge_Pcp69
...,...,...,...,...
95,Total pickup weight_95,10831.7,Pcp37,Lodge_Pcp37
96,Total pickup weight_96,11362.6,Pcp75,Lodge_Pcp75
97,Total pickup weight_97,10123.2,Pcp62,Lodge_Pcp62
98,Total pickup weight_98,11828.4,Pcp50,Lodge_Pcp50


In [81]:
PcpTotalW_df.to_csv(ROOT / 'DOCEL' / 'DOCEL_PickupplanTotalWeight.csv',index=False)

In [82]:
expected_rows = {
    "DOCEL_Events.csv": 1500,
    "DOCEL_Truck_Static.csv": 50,
    "DOCEL_Cargo_Static.csv": 20,
    "DOCEL_CargoStock.csv": 100,
    "DOCEL_PickupplanCargo.csv": 100,
    "DOCEL_PickupplanNotrs.csv": 100,
    "DOCEL_PickupplanTotalWeight.csv": 100,
    "DOCEL_TruckCargoID.csv": 200,
    "DOCEL_TruckPickupplanID.csv": 200,
    "DOCEL_TruckScheduledPickupWeight.csv": 200,
    "DOCEL_TruckStatus.csv": 200,
    "DOCEL_TruckWeight.csv": 400,
}

for filename, expected in expected_rows.items():
    path = ROOT / "DOCEL" / filename

    if not path.exists():
        print(f"MISSING: {filename}")
        continue

    df = pd.read_csv(path)
    actual = len(df)

    if actual == expected:
        status = "OK"
    else:
        status = "KHÁC BASELINE"

    print(
        f"{filename}: "
        f"actual={actual}, "
        f"expected={expected}, "
        f"status={status}"
    )

DOCEL_Events.csv: actual=1500, expected=1500, status=OK
DOCEL_Truck_Static.csv: actual=50, expected=50, status=OK
DOCEL_Cargo_Static.csv: actual=20, expected=20, status=OK
DOCEL_CargoStock.csv: actual=100, expected=100, status=OK
DOCEL_PickupplanCargo.csv: actual=100, expected=100, status=OK
DOCEL_PickupplanNotrs.csv: actual=100, expected=100, status=OK
DOCEL_PickupplanTotalWeight.csv: actual=100, expected=100, status=OK
DOCEL_TruckCargoID.csv: actual=200, expected=200, status=OK
DOCEL_TruckPickupplanID.csv: actual=200, expected=200, status=OK
DOCEL_TruckScheduledPickupWeight.csv: actual=200, expected=200, status=OK
DOCEL_TruckStatus.csv: actual=200, expected=200, status=OK
DOCEL_TruckWeight.csv: actual=400, expected=400, status=OK


In [83]:
events = pd.read_csv(
    ROOT / "DOCEL" / "DOCEL_Events.csv"
)

assert len(events) == 1500
assert events["Event_id"].is_unique
assert events["Event_id"].notna().all()
assert events["Timestamp"].notna().all()
assert events["Activity"].notna().all()

event_ids = set(events["Event_id"])

dynamic_files = [
    "DOCEL_CargoStock.csv",
    "DOCEL_PickupplanCargo.csv",
    "DOCEL_PickupplanNotrs.csv",
    "DOCEL_PickupplanTotalWeight.csv",
    "DOCEL_TruckCargoID.csv",
    "DOCEL_TruckPickupplanID.csv",
    "DOCEL_TruckScheduledPickupWeight.csv",
    "DOCEL_TruckStatus.csv",
    "DOCEL_TruckWeight.csv",
]

for filename in dynamic_files:
    df = pd.read_csv(ROOT / "DOCEL" / filename)

    assert df["objectID"].notna().all(), (
        f"{filename}: objectID có null"
    )

    assert df["eventID"].notna().all(), (
        f"{filename}: eventID có null"
    )

    assert df["eventID"].isin(event_ids).all(), (
        f"{filename}: có eventID không tồn tại"
    )

print("DOCEL VALIDATION: PASS")

DOCEL VALIDATION: PASS


In [84]:
from pathlib import Path
import pandas as pd

OUTPUT_DIR = ROOT / "DOCEL"
BASELINE_DIR = ROOT / "DOCEL_baseline"

assert OUTPUT_DIR.exists(), f"Không tìm thấy: {OUTPUT_DIR}"
assert BASELINE_DIR.exists(), f"Không tìm thấy: {BASELINE_DIR}"

output_files = {
    path.name for path in OUTPUT_DIR.glob("*.csv")
}

baseline_files = {
    path.name for path in BASELINE_DIR.glob("*.csv")
}

missing_files = baseline_files - output_files
extra_files = output_files - baseline_files

assert not missing_files, (
    f"Thiếu file so với baseline: {sorted(missing_files)}"
)

assert not extra_files, (
    f"Có file ngoài baseline: {sorted(extra_files)}"
)

comparison_results = []
differences = {}

for filename in sorted(baseline_files):
    baseline_df = pd.read_csv(
        BASELINE_DIR / filename,
        dtype=str,
        keep_default_na=False
    )

    output_df = pd.read_csv(
        OUTPUT_DIR / filename,
        dtype=str,
        keep_default_na=False
    )

    same_columns = list(output_df.columns) == list(baseline_df.columns)
    same_row_count = len(output_df) == len(baseline_df)

    same_content = False

    if same_columns and same_row_count:
        same_content = output_df.equals(baseline_df)

        if not same_content:
            differences[filename] = baseline_df.compare(
                output_df,
                keep_shape=False,
                keep_equal=False,
                result_names=("baseline", "reproduced")
            ).head(20)

    status = (
        "PASS"
        if same_columns and same_row_count and same_content
        else "FAIL"
    )

    comparison_results.append({
        "File": filename,
        "Baseline rows": len(baseline_df),
        "Reproduced rows": len(output_df),
        "Same columns": same_columns,
        "Same content": same_content,
        "Status": status
    })

comparison_report = pd.DataFrame(comparison_results)
display(comparison_report)

,File,Baseline rows,Reproduced rows,Same columns,Same content,Status
0,DOCEL_CargoStock.csv,100,100,True,True,PASS
1,DOCEL_Cargo_Static.csv,20,20,True,True,PASS
2,DOCEL_Events.csv,1500,1500,True,True,PASS
3,DOCEL_PickupplanCargo.csv,100,100,True,True,PASS
4,DOCEL_PickupplanNotrs.csv,100,100,True,True,PASS
5,DOCEL_PickupplanTotalWeight.csv,100,100,True,True,PASS
6,DOCEL_TruckCargoID.csv,200,200,True,True,PASS
7,DOCEL_TruckPickupplanID.csv,200,200,True,True,PASS
8,DOCEL_TruckScheduledPickupWeight.csv,200,200,True,True,PASS
9,DOCEL_TruckStatus.csv,200,200,True,True,PASS
